In [2]:
import numpy as np
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.layers import Dense
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
import kagglehub
import os
import sklearn
from deap import base, creator, tools
import deap
from deap.algorithms import eaSimple
import time
from sklearn.model_selection import cross_val_score

creator.create("Fitness", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.Fitness)

# data_url = "http://lib.stat.cmu.edu/datasets/boston"
# raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)
# x = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
# y = raw_df.values[1::2, 2]
# x, y = datasets.load_boston(True)

In [3]:
path = kagglehub.dataset_download("camnugent/california-housing-prices")

california_housing = pd.read_csv(os.path.join(path, "housing.csv"))

new_total_bedrooms = [value if boolean == False else california_housing["total_bedrooms"].mean() for value, boolean in zip(california_housing["total_bedrooms"], california_housing["total_bedrooms"].isna())]
california_housing["total_bedrooms"] = new_total_bedrooms
california_housing["avg_rooms"] = california_housing["total_rooms"] / california_housing["households"]
california_housing["avg_bedrooms"] = california_housing["total_bedrooms"] / california_housing["households"]
california_housing["avg_occup"] = california_housing["population"] / california_housing["households"]

x_incompleto = california_housing.drop(columns=["median_house_value", "ocean_proximity", "total_rooms", "total_bedrooms", "households"])
standard_transform = sklearn.preprocessing.StandardScaler()
x_incompleto = standard_transform.fit_transform(x_incompleto)
y = california_housing["median_house_value"].to_numpy().reshape(-1, 1)
minmax_transform = sklearn.preprocessing.MinMaxScaler()
y = minmax_transform.fit_transform(y)

dummy_transform = sklearn.preprocessing.OneHotEncoder()
dummy = dummy_transform.fit_transform(california_housing[["ocean_proximity"]]).toarray()
x = np.concat([x_incompleto, dummy], axis=1)

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.4)
y_train = y_train.reshape(-1)
y_test = y_test.reshape(-1)

Using Colab cache for faster access to the 'california-housing-prices' dataset.


# Random Forest

In [4]:
def fib(a, b, n):
    retorno = [a, b]
    for i in range(2, n):
        temp = a + b
        a = b
        b = temp
        retorno.append(b)
    return retorno

def gerar_hiperparametros_rf():
    hiperparametros = {
        "n_estimators": fib(3, 5, 12),
        "criterion": ["squared_error", "friedman_mse", "poisson"],
        "max_depth": fib(2, 3, 8),
        "min_samples_split": [0.2, 0.1, 0.01, 0.001],
        "min_samples_leaf": [0.2, 0.1, 0.01, 0.001],
        "max_features": ["sqrt", "log2", None],
        "max_leaf_nodes": [None] + fib(2, 3, 8)
    }
    conjunto_gerado = []
    for key in hiperparametros.keys():
        i = np.random.randint(len(hiperparametros[key]))
        conjunto_gerado.append(hiperparametros[key][i])
    return conjunto_gerado

toolbox = base.Toolbox()
toolbox.register("individual", tools.initIterate, creator.Individual, gerar_hiperparametros_rf)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

def evaluate(individuo, min_max_tempo_execucao=[0,1], calcular_tempo=True): # loss function <- esse é o critério de avaliação
    x_train_new, x_validation, y_train_new, y_validation = sklearn.model_selection.train_test_split(x_train, y_train, test_size=0.4)
    inicio = time.time()
    rforest = sklearn.ensemble.RandomForestRegressor(n_estimators=individuo[0], criterion=individuo[1], max_depth=individuo[2],
        min_samples_split=individuo[3], min_samples_leaf=individuo[4], max_features=individuo[5], max_leaf_nodes=individuo[6])
    rforest.fit(x_train_new, y_train_new)
    eqm = np.sum((rforest.predict(x_validation) - y_validation)**2)/len(y_validation)
    tempo_execucao = time.time() - inicio
    tempo_normalizado = (tempo_execucao - min_max_tempo_execucao[0]) / (min_max_tempo_execucao[1] - min_max_tempo_execucao[0])
    if calcular_tempo:
        custo = np.sqrt(eqm) * (1 + tempo_normalizado)
        return custo,
    custo = np.sqrt(eqm)
    return custo,

def funcao_mutacao(individuo, indpb):
    individuo_novo = gerar_hiperparametros_rf()
    for i in range(len(individuo)):
        if np.random.uniform(0, 1) < indpb:
            individuo[i] = individuo_novo[i]
    return individuo,

toolbox.register("mate", tools.cxOnePoint)
toolbox.register("mutate", funcao_mutacao, indpb=0.25)
toolbox.register("select", tools.selTournament, tournsize=3)
toolbox.register("evaluate", evaluate)


def eaCustom(population, toolbox, cxpb, mutpb, ngen, stats=None,
             halloffame=None, verbose=__debug__):
    tempos_execucao = []
    tamanho_populacao_inicial = len(population)

    logbook = tools.Logbook()
    logbook.header = ["gen", "nevals"] + (stats.fields if stats else [])

    # Evaluate the individuals with an invalid fitness
    invalid_ind = [ind for ind in population if not ind.fitness.valid]

    fitnesses = []
    for ind in invalid_ind:
            inicio = time.time()
            fitnesses.append(toolbox.evaluate(ind))
            tempos_execucao.append(time.time() - inicio)
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    if halloffame is not None:
        halloffame.update(population)

    record = stats.compile(population) if stats else {}
    logbook.record(gen=0, nevals=len(invalid_ind), **record)
    if verbose:
        print(logbook.stream)

    min_cada_gen = [record["min"]]

    # Begin the generational process
    for gen in range(1, ngen + 1):
        tempos_execucao_gen_atual = []

        # Select the next generation individuals
        numero_individuos_gerados = tamanho_populacao_inicial + gen * aumento_individuos_por_geracao
        offspring = toolbox.select(population, numero_individuos_gerados)

        # Vary the pool of individuals
        offspring = deap.algorithms.varAnd(offspring, toolbox, cxpb, mutpb)

        # Evaluate the individuals with an invalid fitness
        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        # fitnesses = toolbox.map(toolbox.evaluate, invalid_ind)
        fitnesses = []
        tempo_minimo = 0
        tempo_maximo = 60
        min_max_tempo_execucao = [tempo_minimo, tempo_maximo]
        for ind in invalid_ind:
            inicio = time.time()
            fitnesses.append(toolbox.evaluate(ind, min_max_tempo_execucao))
            tempos_execucao_gen_atual.append(time.time() - inicio)
        for ind, fit in zip(invalid_ind, fitnesses):
            ind.fitness.values = fit

        # Update the hall of fame with the generated individuals
        if halloffame is not None:
            halloffame.update(offspring)

        # Replace the current population by the offspring
        population[:] = offspring
        tempos_execucao = tempos_execucao + tempos_execucao_gen_atual
        # tempos_execucao = tempos_execucao_gen_atual # tempos de execução == tempos de execução da geração anterior

        # Append the current generation statistics to the logbook
        record = stats.compile(population) if stats else {}
        min_cada_gen.append(record["min"])
        logbook.record(gen=gen, nevals=len(invalid_ind), **record)
        if verbose:
            print(logbook.stream)
        patience = 4
        if gen >= patience - 1:
            condicoes_early_stopping = [min_cada_gen[-1] >= min_cada_gen[-i] for i in range(2, patience + 1)]
            # print(condicoes_early_stopping)
            if all(condicoes_early_stopping):
                break

    print("Tempo - Média: {}, Mediana: {} DP: {}, Mínimo: {}, Máximo: {}".format(
        np.mean(tempos_execucao), np.median(tempos_execucao), np.std(tempos_execucao, ddof=1), np.min(tempos_execucao), np.max(tempos_execucao)))
    return population, logbook

tamanho_inicial_populacao = 20
aumento_individuos_por_geracao = 5

In [5]:
pop = toolbox.population(n=tamanho_inicial_populacao) # indivíduos na primeira geração

hof_rf = deap.tools.HallOfFame(10) # objeto que contém os melhores indivíduos
stats = tools.Statistics(key=lambda ind: ind.fitness.values)
stats.register("avg", np.mean)
stats.register("std", np.std)
stats.register("min", np.min)
stats.register("max", np.max)
resultado = eaCustom(pop, toolbox, ngen=20, cxpb=1.0, mutpb=0.1, halloffame=hof_rf, stats=stats)

gen	nevals	avg     	std     	min     	max     
0  	20    	0.389201	0.225342	0.135295	0.956715
1  	25    	0.177704	0.0254421	0.128327	0.22506 
2  	30    	0.156116	0.0151653	0.124542	0.17768 
3  	34    	0.142912	0.0200716	0.120525	0.203942
4  	40    	0.128786	0.00901206	0.120151	0.151362
5  	44    	0.12439 	0.00546238	0.117411	0.154763
6  	50    	0.121826	0.00341332	0.11051 	0.129867
7  	54    	0.123121	0.0134105 	0.111513	0.190844
8  	60    	0.119929	0.00301726	0.114068	0.129106
9  	64    	0.120139	0.00618631	0.114204	0.155402
Tempo - Média: 0.6837485210346212, Mediana: 0.23183393478393555 DP: 1.3406073902827642, Mínimo: 0.013358354568481445, Máximo: 16.05309534072876


In [6]:
for individuo in hof_rf:
    rforest = sklearn.ensemble.RandomForestRegressor(n_estimators=individuo[0], criterion=individuo[1], max_depth=individuo[2],
        min_samples_split=individuo[3], min_samples_leaf=individuo[4], max_features=individuo[5], max_leaf_nodes=individuo[6])
    rforest.fit(x_train, y_train)
    eqm = np.sum((rforest.predict(x_test) - y_test)**2)/len(y_test)
    print(individuo, np.sqrt(eqm))

[89, 'friedman_mse', 13, 0.001, 0.001, 'log2', None] 0.1075142455937561
[5, 'friedman_mse', 21, 0.001, 0.001, 'log2', None] 0.11000692291618376
[5, 'squared_error', 13, 0.001, 0.001, None, None] 0.11255477742851205
[5, 'friedman_mse', 21, 0.001, 0.001, None, None] 0.11298246435396345
[5, 'friedman_mse', 13, 0.001, 0.001, 'log2', None] 0.11217287345097988
[5, 'squared_error', 13, 0.001, 0.001, 'log2', None] 0.11104491786228407
[5, 'squared_error', 21, 0.01, 0.001, None, None] 0.11597742133523947
[5, 'friedman_mse', 21, 0.01, 0.001, None, None] 0.11567356504190238
[5, 'poisson', 21, 0.01, 0.001, None, None] 0.11747982543074155
[5, 'squared_error', 21, 0.001, 0.001, 'log2', None] 0.11463186391155313


# KNN

In [7]:
def gerar_hiperparametros_knn():
    hiperparametros = {
        "n_neighbors": [2, 5, 8, 11, 14, 17, 20],
        "weights": ["uniform", "distance"],
        "algorithm": ["auto", "ball_tree", "kd_tree", "brute"],
        "leaf_size": [6, 12, 25, 50, 100, 150, 200],
        "p": [1, 2, 3, 4]
    }
    conjunto_gerado = []
    for key in hiperparametros.keys():
        i = np.random.randint(len(hiperparametros[key]))
        conjunto_gerado.append(hiperparametros[key][i])
    return conjunto_gerado

toolbox = base.Toolbox()
toolbox.register("individual", tools.initIterate, creator.Individual, gerar_hiperparametros_knn)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

def evaluate(individuo, min_max_tempo_execucao=[0,1], calcular_tempo=True): # loss function <- esse é o critério de avaliação
    x_train_new, x_validation, y_train_new, y_validation = sklearn.model_selection.train_test_split(x_train, y_train, test_size=0.4)
    rknn = sklearn.neighbors.KNeighborsRegressor(n_neighbors=individuo[0], weights=individuo[1], algorithm=individuo[2],
        leaf_size=individuo[3], p=individuo[4])
    inicio = time.time()
    rknn.fit(x_train_new, y_train_new)
    eqm = np.sum((rknn.predict(x_validation) - y_validation)**2)/len(y_validation)
    tempo_execucao = time.time() - inicio
    tempo_normalizado = (tempo_execucao - min_max_tempo_execucao[0]) / (min_max_tempo_execucao[1] - min_max_tempo_execucao[0])
    if calcular_tempo:
        custo = np.sqrt(eqm) * (1 + tempo_normalizado)
        return custo,
    custo = np.sqrt(eqm)
    return custo,

def funcao_mutacao(individuo, indpb):
    individuo_novo = gerar_hiperparametros_knn()
    for i in range(len(individuo)):
        if np.random.uniform(0, 1) < indpb:
            individuo[i] = individuo_novo[i]
    return individuo,

toolbox.register("mate", tools.cxOnePoint)
toolbox.register("mutate", funcao_mutacao, indpb=0.25)
toolbox.register("select", tools.selTournament, tournsize=3)
toolbox.register("evaluate", evaluate)

In [8]:
pop = toolbox.population(n=tamanho_inicial_populacao) # indivíduos na primeira geração

hof_knn = deap.tools.HallOfFame(10) # objeto que contém os melhores indivíduos
stats = tools.Statistics(key=lambda ind: ind.fitness.values)
stats.register("avg", np.mean)
stats.register("std", np.std)
stats.register("min", np.min)
stats.register("max", np.max)
resultado = eaCustom(pop, toolbox, ngen=20, cxpb=1.0, mutpb=0.1, halloffame=hof_knn, stats=stats)

gen	nevals	avg     	std     	min     	max    
0  	20    	0.948403	0.631508	0.166906	1.70794
1  	24    	0.145168	0.0344738	0.124139	0.303413
2  	30    	0.130026	0.00484505	0.123253	0.146208
3  	34    	0.12772 	0.00568935	0.123253	0.158453
4  	40    	0.126602	0.00188725	0.122303	0.129902
5  	45    	0.128254	0.00612735	0.121995	0.161868
6  	50    	0.127381	0.00498222	0.12288 	0.159639
7  	54    	0.128032	0.00725742	0.1224  	0.17942 
8  	60    	0.127288	0.00221381	0.123509	0.134666
Tempo - Média: 1.3083709655355673, Mediana: 0.6622254848480225 DP: 2.2222037904732845, Mínimo: 0.22286772727966309, Máximo: 11.16786789894104


In [9]:
for individuo in hof_knn:
    rknn = sklearn.neighbors.KNeighborsRegressor(n_neighbors=individuo[0], weights=individuo[1], algorithm=individuo[2],
        leaf_size=individuo[3], p=individuo[4])
    rknn.fit(x_train, y_train)
    eqm = np.sum((rknn.predict(x_test) - y_test)**2)/len(y_test)
    print(individuo, np.sqrt(eqm))

[20, 'distance', 'kd_tree', 100, 1] 0.12114765500141185
[20, 'distance', 'brute', 50, 1] 0.12114765500141185
[11, 'uniform', 'brute', 50, 1] 0.1214913628103214
[17, 'distance', 'brute', 50, 1] 0.1207708726025205
[20, 'distance', 'brute', 150, 1] 0.12114765500141185
[8, 'distance', 'kd_tree', 150, 1] 0.12131038619307555
[11, 'distance', 'kd_tree', 50, 1] 0.12037975995888699
[8, 'distance', 'brute', 150, 1] 0.12131038619307555
[11, 'distance', 'kd_tree', 150, 1] 0.12037975995888699
[11, 'uniform', 'brute', 150, 1] 0.1214913628103214


# Gradient Boosting

In [10]:
def gerar_hiperparametros_gb():
    hiperparametros = {
        "loss": ["squared_error", "absolute_error", "huber", "quantile"],
        "learning_rate": [0.001, 0.005, 0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5],
        "n_estimators": fib(3, 5, 12),
        "subsample": [1.0, 0.9, 0.8, 0.7, 0.6],
        "criterion": ["friedman_mse", "squared_error"],
        "min_samples_split": [0.2, 0.1, 0.01, 0.001],
        "min_samples_leaf": [0.2, 0.1, 0.01, 0.001],
        "min_weight_fraction_leaf": [0, 0.1, 0.2, 0.3, 0.4, 0.5],
        "max_depth": [2, 4, 8, 16, 32],
        "min_impurity_decrease": fib(0, 1, 13),
        "max_features": ["sqrt", "log2"],
        "alpha": [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
        "max_leaf_nodes": fib(2, 3, 10),
        "warm_start": [True, False],
        "validation_fraction": [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
        "n_iter_no_change": fib(1, 2, 12),
        "ccp_alpha": fib(0, 1, 13)
    }
    conjunto_gerado = []
    for key in hiperparametros.keys():
        i = np.random.randint(len(hiperparametros[key]))
        conjunto_gerado.append(hiperparametros[key][i])
    return conjunto_gerado

toolbox = base.Toolbox()
toolbox.register("individual", tools.initIterate, creator.Individual, gerar_hiperparametros_gb)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

def evaluate(individuo, min_max_tempo_execucao=[0,1], calcular_tempo=True): # loss function <- esse é o critério de avaliação
    x_train_new, x_validation, y_train_new, y_validation = sklearn.model_selection.train_test_split(x_train, y_train, test_size=0.4)
    rgb = sklearn.ensemble.GradientBoostingRegressor(loss=individuo[0], learning_rate=individuo[1], n_estimators=individuo[2],
        subsample=individuo[3], criterion=individuo[4], min_samples_split=individuo[5], min_samples_leaf=individuo[6],
        min_weight_fraction_leaf=individuo[7], max_depth=individuo[8])
    inicio = time.time()
    rgb.fit(x_train_new, y_train_new)
    eqm = np.sum((rgb.predict(x_validation) - y_validation)**2)/len(y_validation)
    tempo_execucao = time.time() - inicio
    tempo_normalizado = (tempo_execucao - min_max_tempo_execucao[0]) / (min_max_tempo_execucao[1] - min_max_tempo_execucao[0])
    if calcular_tempo:
        custo = np.sqrt(eqm) * (1 + tempo_normalizado)
        return custo,
    custo = np.sqrt(eqm)
    return custo,

def funcao_mutacao(individuo, indpb):
    individuo_novo = gerar_hiperparametros_gb()
    for i in range(len(individuo)):
        if np.random.uniform(0, 1) < indpb:
            individuo[i] = individuo_novo[i]
    return individuo,

toolbox.register("mate", tools.cxOnePoint)
toolbox.register("mutate", funcao_mutacao, indpb=0.25)
toolbox.register("select", tools.selTournament, tournsize=3)
toolbox.register("evaluate", evaluate)

In [11]:
pop = toolbox.population(n=tamanho_inicial_populacao) # indivíduos na primeira geração

hof_gb = deap.tools.HallOfFame(10) # objeto que contém os melhores indivíduos
stats = tools.Statistics(key=lambda ind: ind.fitness.values)
stats.register("avg", np.mean)
stats.register("std", np.std)
stats.register("min", np.min)
stats.register("max", np.max)
resultado = eaCustom(pop, toolbox, ngen=20, cxpb=1.0, mutpb=0.1, halloffame=hof_gb, stats=stats)

gen	nevals	avg     	std     	min     	max    
0  	20    	0.431561	0.303688	0.166018	1.18754
1  	24    	0.179171	0.0702236	0.113287	0.374053
2  	30    	0.13077 	0.0138024	0.115173	0.163042
3  	34    	0.119363	0.00764852	0.111807	0.159558
4  	40    	0.120084	0.0164535 	0.108824	0.212416
5  	44    	0.117643	0.013028  	0.105226	0.176333
6  	50    	0.117926	0.0224985 	0.106813	0.261933
7  	54    	0.113822	0.0184612 	0.105333	0.24553 
8  	60    	0.116489	0.0290177 	0.105855	0.310827
9  	64    	0.113117	0.0182205 	0.104946	0.234788
10 	70    	0.119876	0.0460816 	0.10379 	0.434007
11 	74    	0.111838	0.0118126 	0.105898	0.189006
Tempo - Média: 1.4338298114479011, Mediana: 1.3960105180740356 DP: 1.0022568961713474, Mínimo: 0.02609729766845703, Máximo: 8.912380933761597


In [12]:
for individuo in hof_gb:
    rgb = sklearn.ensemble.GradientBoostingRegressor(loss=individuo[0], learning_rate=individuo[1], n_estimators=individuo[2],
        subsample=individuo[3], criterion=individuo[4], min_samples_split=individuo[5], min_samples_leaf=individuo[6],
        min_weight_fraction_leaf=individuo[7], max_depth=individuo[8])
    rgb.fit(x_train, y_train)
    eqm = np.sum((rgb.predict(x_test) - y_test)**2)/len(y_test)
    print(individuo, np.sqrt(eqm))

['squared_error', 0.3, 55, 0.8, 'friedman_mse', 0.001, 0.001, 0, 4, 1, 'log2', 0.4, 21, True, 0.7, 21, 13] 0.10192657117644634
['squared_error', 0.4, 55, 0.7, 'friedman_mse', 0.001, 0.001, 0, 4, 89, 'sqrt', 0.4, 3, True, 0.1, 233, 8] 0.10426886490927674
['squared_error', 0.3, 55, 0.7, 'friedman_mse', 0.001, 0.01, 0, 16, 3, 'log2', 0.4, 5, False, 0.1, 233, 8] 0.09904872565001611
['squared_error', 0.3, 55, 0.7, 'friedman_mse', 0.001, 0.001, 0, 4, 89, 'log2', 0.4, 55, False, 0.1, 233, 8] 0.10263051078474324
['squared_error', 0.3, 55, 0.7, 'friedman_mse', 0.001, 0.001, 0, 4, 89, 'sqrt', 0.4, 3, True, 0.1, 233, 8] 0.1027492607364282
['squared_error', 0.3, 55, 0.7, 'friedman_mse', 0.001, 0.001, 0, 4, 89, 'sqrt', 0.5, 55, True, 0.3, 8, 55] 0.10273060764896685
['squared_error', 0.3, 55, 0.7, 'friedman_mse', 0.001, 0.001, 0, 4, 89, 'sqrt', 0.5, 55, True, 0.8, 144, 89] 0.10357261946068753
['squared_error', 0.4, 55, 0.7, 'friedman_mse', 0.001, 0.001, 0, 4, 89, 'sqrt', 0.5, 5, True, 0.8, 144, 89] 

## Cross-validation dos melhores modelos - Seleção final

In [13]:
modelo = []
hiperparametros = []
media_cv = []
dp_cv = []
scorer_wrapper = lambda estimador, x, y: np.sqrt(sklearn.metrics.mean_squared_error(estimador.predict(x), y))
for ind in hof_rf:
    rf = sklearn.ensemble.RandomForestRegressor(n_estimators=ind[0], criterion=ind[1], max_depth=ind[2],
        min_samples_split=ind[3], min_samples_leaf=ind[4], max_features=ind[5], max_leaf_nodes=ind[6])
    rf_cvs = sklearn.model_selection.cross_val_score(rf, x, y.flatten(), cv=5, scoring=scorer_wrapper)
    modelo.append("rf")
    hiperparametros.append(ind)
    media_cv.append(np.mean(rf_cvs))
    dp_cv.append(np.std(rf_cvs))
for ind in hof_knn:
    knn = sklearn.neighbors.KNeighborsRegressor(n_neighbors=ind[0], weights=ind[1], algorithm=ind[2],
        leaf_size=ind[3], p=ind[4])
    knn_cvs = sklearn.model_selection.cross_val_score(knn, x, y.flatten(), cv=5, scoring=scorer_wrapper)
    modelo.append("knn")
    hiperparametros.append(ind)
    media_cv.append(np.mean(knn_cvs))
    dp_cv.append(np.std(knn_cvs))
for ind in hof_gb:
    gb = sklearn.ensemble.GradientBoostingRegressor(loss=ind[0], learning_rate=ind[1], n_estimators=ind[2],
        subsample=ind[3], criterion=ind[4], min_samples_split=ind[5], min_samples_leaf=ind[6],
        min_weight_fraction_leaf=ind[7], max_depth=ind[8])
    gb_cvs = sklearn.model_selection.cross_val_score(gb, x, y.flatten(), cv=5, scoring=scorer_wrapper)
    modelo.append("gb")
    hiperparametros.append(ind)
    media_cv.append(np.mean(gb_cvs))
    dp_cv.append(np.std(gb_cvs))

melhores_modelos = pd.DataFrame({"modelo": modelo, "hiperparametros": hiperparametros, "media_cv": media_cv, "dp_cv": dp_cv}).sort_values(by="media_cv")
melhores_modelos

,modelo,hiperparametros,media_cv,dp_cv
28,gb,"[squared_error, 0.3, 55, 0.7, squared_error, 0...",0.130159,0.003985
29,gb,"[squared_error, 0.3, 55, 0.7, squared_error, 0...",0.130294,0.002832
22,gb,"[squared_error, 0.3, 55, 0.7, friedman_mse, 0....",0.130537,0.004226
24,gb,"[squared_error, 0.3, 55, 0.7, friedman_mse, 0....",0.131343,0.004668
27,gb,"[squared_error, 0.4, 55, 0.7, friedman_mse, 0....",0.131827,0.003879
23,gb,"[squared_error, 0.3, 55, 0.7, friedman_mse, 0....",0.133016,0.004576
26,gb,"[squared_error, 0.3, 55, 0.7, friedman_mse, 0....",0.134087,0.005909
21,gb,"[squared_error, 0.4, 55, 0.7, friedman_mse, 0....",0.134198,0.003517
25,gb,"[squared_error, 0.3, 55, 0.7, friedman_mse, 0....",0.135367,0.008556
20,gb,"[squared_error, 0.3, 55, 0.8, friedman_mse, 0....",0.137509,0.009919
